# Start coding and Enjoy the journey

## Install the repo

In [ ]:
!git clone --recursive https://github.com/FarInHeight/Visual-Place-Recognition-Project.git

## (Optional) Mount Google Drive

Useful on Colab to read/write predictions, inliers, CSVs and models across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install dependencies

In [ ]:
%cd Visual-Place-Recognition-Project/image-matching-models
!pip install -e .[all]

In [ ]:
!pip install faiss-cpu

In [ ]:
# required by the retrieval step
!pip install loguru

## Download Datasets

In [ ]:
!python download_datasets.py

## Run your First VPR Evalutation



In [ ]:
!python VPR-methods-evaluation/main.py \
--num_workers 8 \
--batch_size 32 \
--log_dir log_dir \
--method=cosplace --backbone=ResNet18 --descriptors_dimension=512 \
--image_size 512 512 \
--database_folder '<path-to-database-folder>' \
--queries_folder '<path-to-queries-folder>' \
--num_preds_to_save 20 \
--recall_values 1 5 10 20 \
--save_for_uncertainty

## Run Image Matching on Retrieval Results



In [ ]:
!python match_queries_preds.py \
--preds-dir '<path-to-predictions-folder>' \
--matcher 'superpoint-lg' \
--device 'cuda' \
--num-preds 20

## Check Re-ranking Performance

In [ ]:
!python reranking.py \
--preds-dir '<path-to-predictions-folder>' \
--inliers-dir '<path-to-inliers-folder>' \
--num-preds 20 \
--recall-values 1 5 10 20

## Perform Uncertainty Evalutation

In [ ]:
!python -m vpr_uncertainty.eval \
--preds-dir '<path-to-predictions-folder>' \
--inliers-dir '<path-to-inliers-folder>' \
--z-data-path '<path-to-z-data-file>'

# Adaptive Re-ranking (extension)

Full re-ranking runs image matching (IM) on **all** top-20 candidates of every query, which is expensive. **Adaptive re-ranking** first runs IM only on the **top-1** candidate and, from that cheap signal, decides **per query** whether it is worth re-ranking the full top-20 or keeping the retrieval order.

The decision variable is `num_inliers_top1`, except for the SU methods which also use the retrieval L2 distances (`z_data.torch`, saved by retrieval with `--save_for_uncertainty`).

Available methods (`--threshold`): `youden`, `best_r1`, `efficiency` (hard-threshold, no training), `local`, `logistic_hard`, `logistic_help`, `logistic_cost_sensitive`, `sequential` (logistic, need training), `su`, `su_inliers` (need L2 distances).

Pipeline: retrieval → IM top-20 → candidate-level CSV (train + validation split) → training → validation (thresholds) → deploy → check performance.

See the [README](https://github.com/tommasopantano01/Visual-Place-Recognition-Project#adaptive-re-ranking-extension) for the full description of each method.

## 1. Build the candidate-level CSV

Turns the retrieval `.txt` predictions and the top-20 IM `.torch` files into one row per (query, candidate). Run it **twice**: once on the **train** split and once on the **validation** split.

`--z_data_path` is optional and adds the `l2_distance` column: it is required **only** to calibrate the SU methods (`su`, `su_inliers`).

In [ ]:
!python VPR-Adaptive-ReRanking/training/candidate_level/build_candidate_level_csv.py \
--preds_dir '<path-to-predictions-folder>' \
--match_dir '<path-to-top20-inliers-folder>' \
--output_csv '<path-to-candidate_level.csv>' \
--z_data_path '<path-to-z_data.torch>' \
--k 20

## 2. Training (regressor-based methods only)

Trains the single-feature logistic regressors (`hard`, `help`, `hurt`) on the **train** candidate-level CSV and serializes their weights to a `model.json`. The hard-threshold methods (`youden`, `best_r1`, `efficiency`) have **no training step**.

In [ ]:
!python VPR-Adaptive-ReRanking/training/su.py \
--train-csv '<path-to-train-candidate_level.csv>'

## 3. Validation — choose the thresholds

Each method has its own folder `VPR-Adaptive-ReRanking/validation/<method>/`. Validation runs on the **validation** candidate-level CSV and writes the calibrated thresholds (`threshold_<model>_<matcher>.csv`); the deploy step reads them back from here.

**Hard-threshold** methods (`youden`, `best_r1`, `efficiency`) need no model:

In [ ]:
!python VPR-Adaptive-ReRanking/validation/youden/youden.py \
--val-csv '<path-to-validation-candidate_level.csv>' \
--model 'cosplace' \
--matcher 'superpoint-lg'

**Logistic / SU** methods load the trained regressor from `model.json` and grid-search the threshold that maximises the adaptive R@1:

In [ ]:
!python VPR-Adaptive-ReRanking/validation/logistic_hard/logistic_hard.py \
--val-csv '<path-to-validation-candidate_level.csv>' \
--model-json '<path-to-model.json>' \
--model 'cosplace' \
--matcher 'superpoint-lg'

## 4. Run Adaptive Re-ranking (deploy)

`adaptive_reranking.py` selects the method with `--threshold` and runs it live: IM on top-1, per-query decision, and full top-20 IM only for the queries that need it. Each method loads its calibrated threshold from `validation/<method>/` according to `--model` and `--matcher`.

The output folder is organised by stopping budget (`top1/`, `top5/`, `top10/`, `top20/`): each query ends up in exactly one of them.

In [ ]:
!python VPR-Adaptive-ReRanking/adaptive_reranking.py \
--threshold 'youden' \
--model 'cosplace' \
--matcher 'superpoint-lg' \
--preds-dir '<path-to-predictions-folder>' \
--output-dir '<path-to-output-folder>' \
--device 'cuda' \
--num-preds 20
# SU methods (su, su_inliers) additionally need:  --z-data '<path-to-z_data.torch>'

## 5. Check Adaptive Re-ranking Performance

Reads the `top{K}/` folders, reports how many queries stopped at each budget and computes the adaptive recall@N.

In [ ]:
!python VPR-Adaptive-ReRanking/check_performance.py \
--preds-dir '<path-to-predictions-folder>' \
--adaptive-RR-dir '<path-to-output-folder>' \
--num-preds 20 \
--recall-values 1 5 10 20